In [1]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, SplineTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

RANDOM_STATE = 42
TARGET = "Condition_Class"
TRAIN_PATH = "../training_image_features_grouped.csv"
VAL_PATH = "../validation_image_features_grouped.csv"
TEST_PATH = "../testing_image_features_grouped.csv"

EXCLUDE_COLUMNS = [
    "Condition_Class", "Condition_Score", "Damage_Score",
    "Structural_Integrity", "Recovery_Pathway",
    "Reconditioning_Required", "Recovery_Potential",
    "Alternative_Use_Category", "Alternative_Use_Suitability",
    "Sample_ID", "Image_ID"
]

In [2]:
# Shared grouped splits + the agreed 30 features (see codd_common.py).
# Never regenerate the split here: all models must use the same files.
from codd_common import load_grouped_splits, assert_feature_set

data = load_grouped_splits()
train_df, val_df, test_df = data.train_df, data.val_df, data.test_df
X_train, X_val, X_test = data.X_train, data.X_val, data.X_test
y_train, y_val, y_test = data.y_train, data.y_val, data.y_test
feature_columns = data.feature_columns
numeric_features = data.numeric_features
categorical_features = data.categorical_features

assert_feature_set(X_train.columns)
assert_feature_set(X_val.columns)
assert_feature_set(X_test.columns)
print("Features used:", feature_columns)

print('\nTraining class distribution:')
print(y_train.value_counts())


Grouped splits: train (11465, 30) val (2410, 30) test (2644, 30)
Image_ID overlap (must all be 0): {'train-val': 0, 'train-test': 0, 'val-test': 0}
Features: 30 (24 numeric, 6 categorical)
Features used: ['Material_Type', 'Material_Subtype', 'Source', 'Exposure_Type', 'Crack_Level', 'Surface_Damage', 'Breakage_Level', 'Discoloration_Level', 'Deformation_Level', 'Missing_Portion', 'pixel_obj_width', 'pixel_obj_height', 'relative_area', 'edge_density', 'fill_ratio', 'color_std', 'mean_brightness', 'std_brightness', 'contrast', 'texture_laplacian_var', 'dark_pixel_ratio', 'bright_pixel_ratio', 'mean_hue', 'mean_saturation', 'saturation_std', 'contour_area', 'contour_perimeter', 'circularity', 'solidity', 'extent']

Training class distribution:
Condition_Class
Fair         7899
Good         2460
Poor          978
Excellent     128
Name: count, dtype: int64


In [3]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

def make_preprocessor(n_knots=4):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("splines", SplineTransformer(
            n_knots=n_knots, degree=3, knots="quantile", include_bias=False
        )),
        ("scale", StandardScaler())
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer([
        ("numeric_splines", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ], remainder="drop")

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 24
Categorical features: 6


In [4]:
# Six configurations; substantially lighter than the earlier MARS search.
KNOT_OPTIONS = [3, 5]
C_OPTIONS = [0.1, 1.0, 10.0]

validation_rows, candidates = [], {}
for n_knots in KNOT_OPTIONS:
    for C in C_OPTIONS:
        print(f"Training GAM: n_knots={n_knots}, C={C}")
        pipe = Pipeline([
            ("preprocessor", make_preprocessor(n_knots)),
            ("classifier", LogisticRegression(
                penalty="l2", solver="lbfgs", C=C,
                class_weight="balanced", max_iter=1000, tol=1e-3,
                random_state=RANDOM_STATE
            ))
        ])
        start = time.time()
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_val)
        elapsed = time.time() - start
        score = f1_score(y_val, pred, average="macro", zero_division=0)
        key = (n_knots, C)
        candidates[key] = pipe
        validation_rows.append({
            "n_knots": n_knots, "C": C,
            "validation_accuracy": accuracy_score(y_val, pred),
            "validation_macro_f1": score,
            "validation_weighted_f1": f1_score(y_val, pred, average="weighted", zero_division=0),
            "fit_seconds": round(elapsed, 2)
        })

validation_results = pd.DataFrame(validation_rows).sort_values(
    "validation_macro_f1", ascending=False
).reset_index(drop=True)
display(validation_results)
best = validation_results.iloc[0]
best_key = (int(best["n_knots"]), float(best["C"]))
print("Selected configuration:", best_key)

Training GAM: n_knots=3, C=0.1


C:\Users\Ryan John\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Training GAM: n_knots=3, C=1.0


C:\Users\Ryan John\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Training GAM: n_knots=3, C=10.0


C:\Users\Ryan John\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


... (4 outputs left out)

Training GAM: n_knots=5, C=10.0


C:\Users\Ryan John\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,n_knots,C,validation_accuracy,validation_macro_f1,validation_weighted_f1,fit_seconds
0,3,10.0,0.997510,0.981443,0.997551,0.55
1,3,1.0,0.994191,0.968623,0.994302,0.52
2,5,10.0,0.995436,0.964240,0.995540,0.71
3,5,1.0,0.992531,0.957425,0.992688,0.42
4,3,0.1,0.988797,0.955485,0.989023,2.91
5,5,0.1,0.987967,0.935178,0.988513,0.81


Selected configuration: (3, 10.0)


In [5]:
best_n_knots, best_C = best_key
X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)

final_gam = Pipeline([
    ("preprocessor", make_preprocessor(best_n_knots)),
    ("classifier", LogisticRegression(
        penalty="l2", solver="lbfgs", C=best_C,
        class_weight="balanced", max_iter=1000, tol=1e-3,
        random_state=RANDOM_STATE
    ))
])
start = time.time()
final_gam.fit(X_trainval, y_trainval)
print(f"Final GAM training time: {time.time() - start:.2f} seconds")

C:\Users\Ryan John\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Final GAM training time: 0.49 seconds


In [6]:
y_pred = final_gam.predict(X_test)
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision (Macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "Recall (Macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "Macro F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "Weighted F1": f1_score(y_test, y_pred, average="weighted", zero_division=0)
}
print("GAM — Test Metrics")
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

GAM — Test Metrics
Accuracy: 0.9962
Precision (Macro): 0.9676
Recall (Macro): 0.9898
Macro F1: 0.9782
Weighted F1: 0.9963

Classification Report:
              precision    recall  f1-score   support

   Excellent       0.89      0.97      0.93        32
        Fair       1.00      1.00      1.00      1788
        Good       0.99      0.99      0.99       586
        Poor       0.99      1.00      1.00       238

    accuracy                           1.00      2644
   macro avg       0.97      0.99      0.98      2644
weighted avg       1.00      1.00      1.00      2644



In [7]:
labels = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred)])))
cm = confusion_matrix(y_test, y_pred, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title("GAM — Test Confusion Matrix")
plt.tight_layout()
plt.show()

C:\Users\Ryan John\AppData\Local\Temp\ipykernel_37532\1856276638.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
os.makedirs("saved_models", exist_ok=True)
os.makedirs("gam_results", exist_ok=True)
joblib.dump(final_gam, "saved_models/codd_gam_pipeline.joblib")
pd.DataFrame([metrics]).to_csv("gam_results/gam_test_metrics.csv", index=False)
validation_results.to_csv("gam_results/gam_validation_results.csv", index=False)
print("Saved model: saved_models/codd_gam_pipeline.joblib")
print("Saved metrics and validation results in gam_results/")

Saved model: saved_models/codd_gam_pipeline.joblib
Saved metrics and validation results in gam_results/
